# Pre-training segmentation model on ARCADE dataset

In [ ]:
import random

import numpy as np
import torch
from torch.utils.data import DataLoader

We've preparde the `ArcadeSyntaxBinaryDataset` which binaryzes Syntax masks from ARCADE dataset. Now we can pre-train our segmentation model on this dataset. We have also prepared few implementations of segmentation models in `src/segmentation/models` directory. We will use `UNet` architecture for pre-training:
- `CoronaryUNetPP` - wrapper around `Unet++` network from `segmentation_models_pytorch` library.
- `CoronaryDeeplabV3` - wrapper around `DeepLabV3` network from `segmentation_models_pytorch` library.
- `CoronaryUNetCustom` - custom implementation of `UNet` architecture.

In [2]:
from coronary_analysis.models.segmentation import (
    CoronaryUNetPP,
    CoronaryDeeplabV3Plus,
    CoronaryUNetCustom,
)
from coronary_analysis.transforms import (
    get_train_transforms,
    get_val_transforms,
)
from coronary_analysis.datasets import ArcadeSyntaxBinaryDataset
from coronary_analysis.metrics import BCEDiceClDiceCriterion
from coronary_analysis.train import training_loop

In [3]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

device

device(type='cuda')

In [4]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if device.type == "cuda":
        torch.cuda.manual_seed_all(seed)
    elif device.type == "mps":
        torch.mps.manual_seed(seed)

In [5]:
set_seed(42)

We will pre-train the segmentation on images of size `320x320`.

In [6]:
IMG_SIZE = 320

In [7]:
IMAGE_DIR = "raw_datasets/arcade/arcade/syntax"

arcade_train_ds = ArcadeSyntaxBinaryDataset(
    root=IMAGE_DIR,
    split="train",
    transform=get_train_transforms(IMG_SIZE),
)
arcade_val_ds = ArcadeSyntaxBinaryDataset(
    root=IMAGE_DIR,
    split="val",
    transform=get_val_transforms(IMG_SIZE),
)

loading annotations into memory...
Done (t=0.21s)
creating index...
index created!
loading annotations into memory...
Done (t=0.03s)
creating index...
index created!


We've prepared 10 configuration for pre-training models, which then will be used for fine-tuning.

In [ ]:
configs = [
    {
        "name": "CoronaryUNetCustom_0",
        "model": CoronaryUNetCustom(
            "resnet34", encoder_weights="imagenet", dropout=0.2
        ),
        "optimizer": torch.optim.Adam,
        "criterion": BCEDiceClDiceCriterion(0.5, 0.5, 0.0),
        "lr": 1e-3,
        "batch_size": 32,
        "num_epochs": 50,
    },
    {
        "name": "CoronaryUNetCustom_1",
        "model": CoronaryUNetCustom(
            "resnet34", encoder_weights=None, dropout=0.3, depth=5
        ),
        "optimizer": torch.optim.Adam,
        "criterion": BCEDiceClDiceCriterion(0.5, 0.5, 0.0),
        "lr": 1e-3,
        "batch_size": 32,
        "num_epochs": 50,
    },
    {
        "name": "CoronaryUNetCustom_2",
        "model": CoronaryUNetCustom(
            "resnet34", encoder_weights=None, dropout=0.2, depth=4
        ),
        "optimizer": torch.optim.Adam,
        "criterion": BCEDiceClDiceCriterion(0.5, 0.5, 0.0),
        "lr": 1e-3,
        "batch_size": 32,
        "num_epochs": 50,
    },
    {
        "name": "CoronaryUNetCustom_3",
        "model": CoronaryUNetCustom(
            "resnet18", encoder_weights=None, dropout=0.2, depth=4
        ),
        "optimizer": torch.optim.Adam,
        "criterion": BCEDiceClDiceCriterion(0.5, 0.5, 0.0),
        "lr": 1e-3,
        "batch_size": 32,
        "num_epochs": 100,
    },
    {
        "name": "CoronaryUNetPP_0",
        "model": CoronaryUNetPP("resnet18", encoder_weights="imagenet"),
        "optimizer": torch.optim.Adam,
        "criterion": BCEDiceClDiceCriterion(0.5, 0.5, 0.0),
        "lr": 1e-3,
        "batch_size": 32,
        "num_epochs": 50,
    },
    {
        "name": "CoronaryUNetPP_1",
        "model": CoronaryUNetPP("resnet34", encoder_weights="imagenet"),
        "optimizer": torch.optim.Adam,
        "criterion": BCEDiceClDiceCriterion(0.5, 0.5, 0.0),
        "lr": 1e-3,
        "batch_size": 32,
        "num_epochs": 50,
    },
    {
        "name": "CoronaryUNetPP_2",
        "model": CoronaryUNetPP("resnet34", encoder_weights=None),
        "optimizer": torch.optim.Adam,
        "criterion": BCEDiceClDiceCriterion(0.5, 0.5, 0.0),
        "lr": 1e-3,
        "batch_size": 20,
        "num_epochs": 50,
    },
    {
        "name": "CoronaryDeeplabV3Plus_0",
        "model": CoronaryDeeplabV3Plus("resnet18", encoder_weights="imagenet"),
        "optimizer": torch.optim.Adam,
        "criterion": BCEDiceClDiceCriterion(0.5, 0.5, 0.0),
        "lr": 1e-3,
        "batch_size": 32,
        "num_epochs": 50,
    },
    {
        "name": "CoronaryDeeplabV3Plus_1",
        "model": CoronaryDeeplabV3Plus("resnet18", encoder_weights=None),
        "optimizer": torch.optim.Adam,
        "criterion": BCEDiceClDiceCriterion(0.5, 0.5, 0.0),
        "lr": 1e-3,
        "batch_size": 32,
        "num_epochs": 50,
    },
]

In [9]:
import os

PRETRAINED_MODELS_DIR = "../models/pretrained/arcade_syntax_binary"

if not os.path.exists(PRETRAINED_MODELS_DIR):
    os.makedirs(PRETRAINED_MODELS_DIR)

Train all configurations and save pre-trained weights for fine-tuning.

In [ ]:
for i, config in enumerate(configs):
    if device.type == "cuda":
        torch.cuda.empty_cache()
    elif device.type == "mps":
        torch.mps.empty_cache()

    file_name = f"{PRETRAINED_MODELS_DIR}/{config['name']}.pth"
    model = config["model"].to(device)
    optimizer = config["optimizer"](model.parameters(), lr=config["lr"])
    criterion = config["criterion"]
    batch_size = config["batch_size"]
    num_epochs = config["num_epochs"]

    train_loader = DataLoader(arcade_train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(arcade_val_ds, batch_size=batch_size, shuffle=False)

    training_loop(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=optimizer,
        criterion=criterion,
        device=device,
        num_epochs=num_epochs,
    )

    torch.save(model.state_dict(), file_name)